### Tratamento de dados para Modelo de LLM Textual Especializado em Psicologia Educacional


<p>Abaixo está os diagramas do sistema de LLM Textual, com uma representão gráfica dos processos utilizados para modelagem e arquitetura inicial para treinamento e Especialização do modelo de Inteligência Artificial, sendo considerado um trabalho que levou aos primeiros passos essenciais para rodar tal projeto. </p>

<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding: 15px; border: 1px solid #ddd; border-radius: 8px;">
      <h3 style="margin-top: 0;">Pipeline Técnico</h3>
      <p>O fluxo de processamento de dados (RAG) é composto por:</p>
      <ul>
        <li><b>Ingestão:</b> Extração de PDF (PyPDFLoader).</li>
        <li><b>Chunking:</b> 938 segmentos de 500 tokens.</li>
        <li><b>Vetorização:</b> Armazenamento em ChromaDB.</li>
      </ul>
      <p><i>Etapa 1 concluída com sucesso.</i></p>
    </td>
    <td style="width: 50%; vertical-align: top; padding: 15px; border: 1px solid #ddd; border-radius: 8px;">
      <h3 style="margin-top: 0;">Diagrama de Arquitetura</h3>
      <img src="../images/fluxograma_vertical.png" alt="Fluxo" style="width: 100%;">
      <p style="text-align: center; font-size: 12px;">Figura 1: Representação do pipeline.</p>
    </td>
  </tr>
</table>

## Instalações de Dependências

<p>Para dar início, foi feito as instalações por meio do comando python de forma a baixar as tecnologias</p>
<div align="center">

### 🛠 Tecnologias e Bibliotecas Utilizadas

#### Core & Fine-Tuning
<a href="#"><img src="https://img.shields.io/badge/Python-3.10+-3776AB?style=for-the-badge&logo=python&logoColor=white" alt="Python"></a>
<a href="#"><img src="https://img.shields.io/badge/PyTorch-EE4C2C?style=for-the-badge&logo=pytorch&logoColor=white" alt="PyTorch"></a>
<a href="#"><img src="https://img.shields.io/badge/HuggingFace-FFD21E?style=for-the-badge&logo=huggingface&logoColor=black" alt="HuggingFace"></a>
<a href="#"><img src="https://img.shields.io/badge/PEFT-000000?style=for-the-badge&logo=huggingface&logoColor=white" alt="PEFT"></a>
<a href="#"><img src="https://img.shields.io/badge/BitsAndBytes-FF9900?style=for-the-badge&logo=nvidia&logoColor=white" alt="BitsAndBytes"></a>

#### RAG & Dados
<a href="#"><img src="https://img.shields.io/badge/LangChain-1C3C3C?style=for-the-badge&logo=langchain&logoColor=white" alt="LangChain"></a>
<a href="#"><img src="https://img.shields.io/badge/ChromaDB-8B4513?style=for-the-badge&logo=chromadb&logoColor=white" alt="ChromaDB"></a>
<a href="#"><img src="https://img.shields.io/badge/SentenceTransformers-00A67D?style=for-the-badge&logo=python&logoColor=white" alt="SentenceTransformers"></a>
<a href="#"><img src="https://img.shields.io/badge/PyPDF-F40F02?style=for-the-badge&logo=adobeacrobatreader&logoColor=white" alt="PyPDF"></a>

#### API & Backend
<a href="#"><img src="https://img.shields.io/badge/FastAPI-05998B?style=for-the-badge&logo=fastapi&logoColor=white" alt="FastAPI"></a>
<a href="#"><img src="https://img.shields.io/badge/Uvicorn-000000?style=for-the-badge&logo=python&logoColor=white" alt="Uvicorn"></a>
<a href="#"><img src="https://img.shields.io/badge/Pydantic-E92063?style=for-the-badge&logo=pydantic&logoColor=white" alt="Pydantic"></a>

#### Avaliação
<a href="#"><img src="https://img.shields.io/badge/Perplexity-6A5ACD?style=for-the-badge&logo=openai&logoColor=white" alt="Perplexity"></a>
<a href="#"><img src="https://img.shields.io/badge/ROUGE-228B22?style=for-the-badge&logo=python&logoColor=white" alt="RougeScore"></a>

</div>

## Esse procedimento foi feito de `forma incessante`, de forma que todos os passos foram interligados com sucesso. Me enganei no início ao pensar que estava usando um modelo para treinamento que podia ser usado, mas a minha surpresa era que a quantidade de parâmetros da tecnologia era muito baixa.

In [ ]:
!pip install -r requirements.txt

### Carregamento e Divisão do Documento

<p>O primeiro passo é fazer um tratamento possível que traga o pdf para uma análise mais precisa do procedimento que realizei, sendo essencial para o início dos requerimentos dos chunks necessários, começando a partir da próxima cédula</p>

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Carregamento do PDF
loader = PyPDFLoader("./docs/20260042690.pdf")
docs = loader.load()

# Definindo a estratégia de Chunking
# Teste com 500 e depois 1000 para comparar os resultados
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(docs)

print(f"Total de chunks gerados: {len(chunks)}")

### Criar Banco de dados Vetorial

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Definir o modelo de Embedding (usaremos um modelo leve do HuggingFace)
# Este modelo é ideal para rodar localmente no ambiente de estudos
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Criar o Banco Vetorial a partir dos chunks
# O Chroma vai processar os 938 chunks e criar a indexação vetorial
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db" # Isso salva os dados em disco para você não perder
)

print("Banco vetorial criado com sucesso!")

In [ ]:
# Teste de recuperação
query = "O que é Psicologia Educacional?"
docs_recuperados = vectorstore.similarity_search(query, k=2)

print(f"Trecho encontrado: {docs_recuperados[0].page_content}")

In [ ]:
import json

dataset = []

# Iterando sobre uma amostra ou sobre todos os chunks
for i, chunk in enumerate(chunks[:100]): # Teste com 100 para começar
    # Defina aqui o seu prompt de geração (persona de Psicologia Educacional)
    prompt = f"""
    Você é um especialista em Psicologia Educacional. Analise o texto abaixo e elabore:
    1. Uma pergunta desafiadora baseada no conteúdo.
    2. Uma resposta técnica e fundamentada no texto.

    Formate como JSON: {{"Instruction": "Pergunta", "Output": "Resposta"}}

    Texto: {chunk.page_content}
    """

    # Aqui você chamaria o seu modelo (ex: client.chat.completions.create(...))
    # resposta = chamar_modelo(prompt)
    # dataset.append(json.loads(resposta))

# Salvar o dataset final
with open("./dataset/dataset_gerado.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Dataset gerado com sucesso!")

Com isso, escolhi o modelo `Qwen/Qwen2.5-3B-Instruct`, que possui `3 bilhões de parâmetros`, algo que reduziria e muito a alucinação do modelo de Inteligência Artificial, e assim segui, contornei o modelo antigo que não poderia ser usado pois apresentou muita interferência, por um modelo que me traria muito mais informações de ponta a ponta, sem falha e com qualidade. 

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Criar o pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=150)
llm = HuggingFacePipeline(pipeline=pipe)

print("Modelo carregado com sucesso!")

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Carregar modelo otimizado
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Pipeline com configurações de performance
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,             # Reduzi de 150 para 50 (suficiente para instruções curtas)
    pad_token_id=50256,            # Resolve o warning de pad_token
    clean_up_tokenization_spaces=False # Resolve o warning de BPE
)
llm = HuggingFacePipeline(pipeline=pipe)

## Início de toda a CURADORIA MANUAL do Dataset

## O projeto foi seguido por uma curadoria fina dos dados, para ser usado como fonte de perguntas necessárias a serem feitas em requisições GET e POST na API desenvolvida, seguindo uma arquitetura limpa para comparar as respostas da API com os Pares de Perguntas e respostas

In [ ]:
import json

# --- 1. SETUP DO MODELO ---
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    pad_token_id=50256,
    clean_up_tokenization_spaces=False
)
llm = HuggingFacePipeline(pipeline=pipe)

# --- 2. LOOP QUE GERA OS 100 PARES ---
dataset = []

# Vamos iterar sobre os primeiros 100 chunks do seu documento
for i, chunk in enumerate(chunks[:100]):
    print(f"Gerando par {i+1}/100...")

    prompt = f"Instrução: Elabore uma pergunta sobre Psicologia Educacional baseada neste texto: {chunk.page_content}\n\nResposta:"

    try:
        # AQUI o modelo gera o texto
        resposta_completa = llm.invoke(prompt)

        # Limpeza simples: pega apenas o texto após 'Resposta:'
        if "Resposta:" in resposta_completa:
            resposta = resposta_completa.split("Resposta:")[1].strip()
        else:
            resposta = resposta_completa.strip()

        # Adiciona ao dataset
        item = {"Instruction": "Pergunta de Psicologia Educacional", "Output": resposta}
        dataset.append(item)

    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

# --- 3. SALVAR O RESULTADO ---
with open("./dataset/dataset_treino.jsonl", "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Dataset de 100 pares gerado e salvo em './dataset/dataset_gerado.jsonl' com sucesso!")

## Para curadoria, segui o seguinte código

In [3]:
import json

def curadoria_manual_seletiva(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        dataset = [json.loads(line) for line in f]
    
    dataset_curado = []
    
    print(f"Iniciando curadoria de {len(dataset)} registros...")
    
    for i, par in enumerate(dataset):
        # Acessa os dados ignorando campos vazios/genéricos
        instr = par.get('instruction', '') or par.get('Instruction', '')
        out = par.get('output', '') or par.get('Output', '')
        
        if not out or out == 'VAZIO' or 'Educational psychology explains' in out:
            continue
            
        print(f"\n--- Par {i+1} ---")
        print(f"Pergunta: {instr}")
        print(f"Resposta: {out}")
        
        decisao = input("Manter? (s = SIM / qualquer tecla = APAGAR): ").lower()
        if decisao == 's':
            dataset_curado.append(par)
            
    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in dataset_curado:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            
    print(f"\nConcluído! {len(dataset_curado)} pares salvos em {output_file}.")
# Execute com os nomes dos seus arquivos - Usando 
curadoria_manual_seletiva(r'C:\Users\josem\OneDrive\Documentos\AV2IA\dataset\dataset_gerado.jsonl', r'C:\Users\josem\OneDrive\Documentos\AV2IA\dataset\dataset_gerado_curado.jsonl')

Iniciando curadoria de 100 registros...

--- Par 1 ---
Pergunta: Pergunta de Psicologia Educacional
Resposta: 1
Aqui está uma pergunta sobre Psicologia Educacional baseada no texto fornecido:

Com base no conteúdo do texto, qual é a principal função da psicologia educacional na escola e como ela auxilia os

--- Par 2 ---
Pergunta: Pergunta de Psicologia Educacional
Resposta: Aqui está uma sugestão de pergunta baseada no texto sobre Psicologia Educacional:

"Com base na leitura do texto, qual é a importância da compreensão das dinâmicas familiares na abordagem

--- Par 3 ---
Pergunta: Pergunta de Psicologia Educacional
Resposta: Qual é a relação entre as competências socioemocionais e o desempenho escolar dos alunos na perspectiva da psicologia educacional? 

Esta pergunta foi elaborada com base no conteúdo da disciplina de

--- Par 4 ---
Pergunta: Pergunta de Psicologia Educacional
Resposta: Como um modelo de pergunta sobre Psicologia Educacional baseada no texto fornecido, podemos for